# Encoder input, T = 32, all scenario folders, benign only: genuine windows for one-class training

This notebook builds a **benign-only** encoder input from **all 8 scenario folders** (DataReplaySybil,
DoSDisruptiveSybil, DoSRandomSybil, GridSybil × the 0709 / 1416 groups) for one-class (genuine-only) autoencoder
training: the model learns genuine motion from genuine windows only, and attacks are what it reconstructs badly.
It reads the prepared files in `src/data/prepared_data/` (written by `src/pipeline/input_representation.ipynb`).

**Crop.** A _link_ is everything one receiving car heard from one sender pseudonym, in time order. Each link is cut
into consecutive, **non-overlapping** pieces of at most **32 messages** (a link of 70 messages gives 32 + 32 + 6).
Every piece is padded to 32 rows; `mask` is 1 on the real rows (a prefix) and 0 on the padding rows. Every message
of a kept link lands in exactly one window (`min_messages = 1`). 13 features (D3), no time of day.

**Split (⚠ RULE 3: a new split, flagged).** 80 / 10 / 10 train / val / test **by physical vehicle**, with the vehicle
key of `src/pipeline/all/encoder_input_T24.ipynb`: `"<group>:<sender vehicle>"`, one split per key across all 4
families of a group (F1: the 0709 / 1416 families replay the same traffic, so the same car must stay in one split).
Strata = (group, class of the vehicle), where the class is the set of attack codes the key carries over the 4
families (e.g. `Benign`, `DataReplaySybil+DoSRandomSybil+DoSDisruptiveSybil`). In each stratum the T48_tvt rule:
seeded (0) shuffle of the sorted keys, test = the first 10%, val = the next 10%, train = the rest.

**Benign only in train / val / test.** A link is kept only if its sender vehicle has code 0 in its scenario folder
and is never an attacker there (by the `index.json` labels or by the attack code of that vehicle's own trace file,
as in `T48_benign`). What this removes is counted.

**Evaluation splits.** The attack windows of the **test** vehicles go to `test_gridsybil`, `test_datareplay`,
`test_dosrandom`, `test_dosdisruptive` (same T, same format). Attack windows of train / val vehicles are dropped.
DataReplay / DoS attackers change pseudonym every 1–2 messages (F11), so most of their windows hold 1–3 real rows;
the share is reported per split.

**Normalisation.** Mean / std of the real rows of the **benign train** windows, applied to every split.

**Known limitations.** F1: the 4 families of a group replay the same benign traffic, so most benign windows have
near-copies in the other 3 families (all copies kept, each window records its `family`; they are in the same split
because the key is per group). F14: `GridSybil_0709` is a separate re-simulation with different vehicle ids; a shared
id there is not the same car, and its near-copies of cars in the other 0709 families can straddle splits. Not
measured here.

**Memory.** Windows are never all held in memory: a counting pass collects counts and the benign train sums, then a
writing pass normalises and streams the shards to disk.

In [1]:
from __future__ import annotations

import dataclasses
import gzip
import json
import multiprocessing
import os
import pathlib
import platform
import re
import shutil
import subprocess
import time
from collections.abc import Callable, Iterator
from datetime import datetime, timezone
from typing import Any

import numpy as np
import pandas as pd

# Run from the repository root, so that every relative path resolves the same way every time.
for _folder in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:  # notebooks live in src/<folder>/...
    if (_folder / "CLAUDE.md").is_file():
        os.chdir(_folder)
        break
assert (
    pathlib.Path("CLAUDE.md").is_file() and pathlib.Path("data").is_dir()
), f"run this notebook from the repository root (now in {pathlib.Path.cwd()})"
print("working directory:", pathlib.Path.cwd())
_NOTEBOOK_START = time.perf_counter()

working directory: /Users/bibekgupta/Downloads/projects/sybil-llm


## Settings and the disk guard

Everything that can change lives in one frozen dataclass; it is stored in `metadata.json` next to the output.
The output may only go under `src/data/encoder_input/all/` (never under `data/`, RULE 2). `DiskGuard` refuses to
write when the projected output is larger than the free space minus 5 GB.

In [2]:
ALL_SCENARIOS: tuple[str, ...] = (
    "DataReplaySybil_0709",
    "DataReplaySybil_1416",
    "DoSDisruptiveSybil_0709",
    "DoSDisruptiveSybil_1416",
    "DoSRandomSybil_0709",
    "DoSRandomSybil_1416",
    "GridSybil_0709",
    "GridSybil_1416",
)


@dataclasses.dataclass(frozen=True)
class EncoderInputSettings:
    """Settings for building the benign-only T = 32 encoder input over all scenario folders.

    Attributes:
        prepared_dir: Folder of prepared files plus `index.json` (read only here).
        out_dir: Where the shards go (gitignored).
        allowed_out_root: The only folder the writer may write into.
        seq_len: Maximum messages per window (T); every window is padded to this length.
        crop_stride: Messages between two crop starts; equal to `seq_len` means no overlap.
        min_messages: Windows with fewer real messages are dropped (1 keeps every window).
        scenarios: Scenario folders to read.
        benign_code: Attack code of genuine senders; train / val / test keep only these.
        attack_splits: Attack code -> name of the evaluation split holding the attack windows of the test vehicles.
        test_frac: Share of vehicles per stratum that go to test (the first ones of the shuffled stratum).
        val_frac: Share of vehicles per stratum that go to val (the next ones).
        split_seed: Seed of the vehicle shuffle.
        short_below: Windows with fewer real rows than this are reported as very short (1–3 rows).
        shard_size: Windows per output shard.
        decimals: Decimals kept for every float written to disk.
        gzip_above_gb: Shards are gzipped (`.json.gz`) when the projected plain-JSON output exceeds this size.
        workers: Number of worker processes for reading.
        files_per_batch: Prepared files handed to the workers at a time (bounds memory).
        size_sample_files: Prepared files per scenario used to estimate the output size.
        disk_margin_gb: Free space that must remain after writing.
        overwrite: Whether an existing output folder may be replaced.
    """

    prepared_dir: str = "src/data/prepared_data"
    out_dir: str = "src/data/encoder_input/all/T32_benign"
    allowed_out_root: str = "src/data/encoder_input/all"
    seq_len: int = 32
    crop_stride: int = 32
    min_messages: int = 1
    scenarios: tuple[str, ...] = ALL_SCENARIOS
    benign_code: int = 0
    attack_splits: tuple[tuple[int, str], ...] = (
        (16, "test_gridsybil"),
        (17, "test_datareplay"),
        (18, "test_dosrandom"),
        (19, "test_dosdisruptive"),
    )
    test_frac: float = 0.10
    val_frac: float = 0.10
    split_seed: int = 0
    short_below: int = 4
    shard_size: int = 10_000
    decimals: int = 4
    gzip_above_gb: float = 4.0
    workers: int = 10
    files_per_batch: int = 512
    size_sample_files: int = 24
    disk_margin_gb: float = 5.0
    overwrite: bool = True

    def __post_init__(self) -> None:
        if self.seq_len < 1:
            raise ValueError(f"seq_len must be >= 1, got {self.seq_len}")
        if self.crop_stride != self.seq_len:
            raise ValueError("this notebook writes non-overlapping crops only (crop_stride == seq_len)")
        if not 1 <= self.min_messages <= self.seq_len:
            raise ValueError(f"min_messages must be in [1, seq_len], got {self.min_messages}")
        if not set(self.scenarios) <= set(ALL_SCENARIOS) or not self.scenarios:
            raise ValueError(f"unknown scenario folders in {self.scenarios}")
        if not 0.0 < self.test_frac < 1.0 or not 0.0 < self.val_frac < 1.0 or self.test_frac + self.val_frac >= 1.0:
            raise ValueError(
                f"need 0 < test_frac, val_frac and test_frac + val_frac < 1, got {self.test_frac}, {self.val_frac}"
            )
        if self.benign_code in dict(self.attack_splits):
            raise ValueError("benign_code must not be an attack code")
        if self.workers < 1 or self.shard_size < 1 or self.files_per_batch < 1:
            raise ValueError("workers, shard_size and files_per_batch must be >= 1")
        out = pathlib.Path(self.out_dir).resolve()
        if out.is_relative_to(pathlib.Path("data").resolve()):
            raise ValueError("out_dir must not be inside data/ (read-only, RULE 2)")
        if not out.is_relative_to(pathlib.Path(self.allowed_out_root).resolve()):
            raise ValueError(f"out_dir {out} is outside the allowed folder {self.allowed_out_root}")

    @property
    def vehicle_splits(self) -> tuple[str, ...]:
        """The benign splits, which are also the vehicle splits."""
        return ("train", "val", "test")

    @property
    def all_splits(self) -> tuple[str, ...]:
        """Every output split: the benign ones, then the attack evaluation splits."""
        return (*self.vehicle_splits, *(name for _, name in self.attack_splits))


class DiskGuard:
    """Refuses to write when the projected output would leave less than a safety margin of free space."""

    def __init__(self, folder: str | pathlib.Path, margin_gb: float) -> None:
        """Stores the folder whose disk is checked.

        Args:
            folder: Any folder on the target disk (its nearest existing parent is used).
            margin_gb: Free space, in GB, that must remain after writing.
        """
        self.folder = pathlib.Path(folder).resolve()
        self.margin_bytes = int(margin_gb * 1e9)

    def free_bytes(self) -> int:
        """Free bytes on the disk that holds the folder."""
        existing = next(p for p in [self.folder, *self.folder.parents] if p.exists())
        return shutil.disk_usage(existing).free

    def allowance(self) -> int:
        """Bytes that may be written: free space minus the margin."""
        return self.free_bytes() - self.margin_bytes

    def check(self, projected_bytes: float, already_there: int = 0) -> None:
        """Raises if the projected output does not fit.

        Args:
            projected_bytes: Projected size of the output.
            already_there: Bytes of an existing output that will be replaced (freed before writing).
        """
        allowance = self.allowance() + already_there
        if projected_bytes > allowance:
            raise RuntimeError(
                f"projected output {projected_bytes / 1e9:.2f} GB exceeds free space minus "
                f"{self.margin_bytes / 1e9:.0f} GB ({allowance / 1e9:.2f} GB); refusing to write"
            )


def folder_bytes(folder: pathlib.Path) -> int:
    """Total size of the files under a folder (0 if it does not exist)."""
    return sum(p.stat().st_size for p in folder.rglob("*") if p.is_file()) if folder.exists() else 0


SETTINGS = EncoderInputSettings()
FEATURE_NAMES: tuple[str, ...] = (
    "claimed_pos_x",
    "claimed_pos_y",
    "rx_pos_x",
    "rx_pos_y",
    "claimed_vel_x",
    "claimed_vel_y",
    "rx_vel_x",
    "rx_vel_y",
    "claimed_acl_x",
    "claimed_acl_y",
    "range",
    "bearing",
    "log_dtau",
)
N_FEATURES = len(FEATURE_NAMES)
LOG_DTAU_COL = FEATURE_NAMES.index("log_dtau")
GUARD = DiskGuard(SETTINGS.out_dir, SETTINGS.disk_margin_gb)
print(SETTINGS)
print(f"Takeaway: {GUARD.free_bytes() / 1e9:.1f} GB free; the output may use at most {GUARD.allowance() / 1e9:.1f} GB.")

EncoderInputSettings(prepared_dir='src/data/prepared_data', out_dir='src/data/encoder_input/all/T32_benign', allowed_out_root='src/data/encoder_input/all', seq_len=32, crop_stride=32, min_messages=1, scenarios=('DataReplaySybil_0709', 'DataReplaySybil_1416', 'DoSDisruptiveSybil_0709', 'DoSDisruptiveSybil_1416', 'DoSRandomSybil_0709', 'DoSRandomSybil_1416', 'GridSybil_0709', 'GridSybil_1416'), benign_code=0, attack_splits=((16, 'test_gridsybil'), (17, 'test_datareplay'), (18, 'test_dosrandom'), (19, 'test_dosdisruptive')), test_frac=0.1, val_frac=0.1, split_seed=0, short_below=4, shard_size=10000, decimals=4, gzip_above_gb=4.0, workers=10, files_per_batch=512, size_sample_files=24, disk_margin_gb=5.0, overwrite=True)
Takeaway: 119.4 GB free; the output may use at most 114.4 GB.


## Labels, attackers and the 80 / 10 / 10 vehicle split

`IndexBook` reads the attack code of every sender vehicle per scenario folder from `index.json`.

`VehicleSplit` keys a physical vehicle as `"<group>:<vehicle>"` (the T24 key) and gives it one split for all 4
families of its group. Its class is the set of known attack codes it carries over the families (`-1`, unknown, is
ignored; a key with only unknown codes has class `Unknown`). Strata = (group, class), sorted; in each, the sorted
keys are shuffled by one generator seeded with 0 (drawn stratum after stratum, as in `T48_tvt`): test = the first
`max(1, round(0.1 n))`, val = the next `max(1, round(0.1 n))`, train = the rest. The older split stored in
`index.json` is not used.

`AttackerRegistry` lists the attacker vehicles of every scenario folder from two sources: the `index.json` labels and
the attack code in the name of each vehicle's own trace file (`traceJSON-<vehicle>-<pseudonym>-A<code>-…`). The
file code is checked against the `receiver_attack_code` inside the file during the counting pass.

In [3]:
def group_of(scenario: str) -> str:
    """Returns the time-window group of a scenario folder, e.g. 'GridSybil_0709' -> '0709'."""
    return scenario.rsplit("_", 1)[1]


def family_of(scenario: str) -> str:
    """Returns the attack family of a scenario folder, e.g. 'GridSybil_0709' -> 'GridSybil'."""
    return scenario.rsplit("_", 1)[0]


def vehicle_key(scenario: str, vehicle: int) -> str:
    """The physical-vehicle key of the T24 notebook: '<group>:<vehicle>'."""
    return f"{group_of(scenario)}:{vehicle}"


TRACE_NAME = re.compile(r"^traceJSON-(\d+)-(\d+)-A(\d+)-")


class IndexBook:
    """Sender labels and per-scenario totals, read from `index.json`.

    Attributes:
        labels: Scenario folder -> {vehicle id -> attack code}.
        class_names: Attack code -> class name.
        counts: Scenario folder -> class name -> {links, messages, ...} as recorded by the input representation.
        normalisation_mean: Mean recorded in the index (a numerical shift and the size-estimate stand-in only).
        normalisation_std: Standard deviation recorded in the index (size estimate only).
        source: Path and provenance of the index file.
    """

    def __init__(self, index_path: pathlib.Path) -> None:
        """Loads the index.

        Args:
            index_path: Path to `index.json`.
        """
        index = json.loads(index_path.read_text())
        if list(index["features"]) != list(FEATURE_NAMES):
            raise ValueError(f"feature order in {index_path} differs from FEATURE_NAMES")
        self.labels: dict[str, dict[int, int]] = {
            scenario: {int(vehicle): int(code) for vehicle, code in codes.items()}
            for scenario, codes in index["labels"].items()
        }
        self.class_names: dict[int, str] = {int(code): name for code, name in index["class_names"].items()}
        self.counts: dict[str, dict[str, dict[str, int]]] = index["counts"]
        self.normalisation_mean = np.asarray(index["normalisation"]["mean"], dtype=np.float64)
        self.normalisation_std = np.asarray(index["normalisation"]["std"], dtype=np.float64)
        self.source = {"path": str(index_path), "provenance": index.get("provenance", {})}

    def label(self, scenario: str, sender: int) -> int:
        """Returns the sender's attack code, or -1 if it is not known."""
        return self.labels.get(scenario, {}).get(sender, -1)

    def name(self, code: int) -> str:
        """Class name of an attack code."""
        return self.class_names.get(code, str(code))


class VehicleSplit:
    """The 80 / 10 / 10 train / val / test split by physical vehicle, stratified by (group, class of the vehicle).

    Attributes:
        split_of: Vehicle key -> "train", "val" or "test".
        class_of: Vehicle key -> class (its attack codes over the families, e.g. "Benign").
        report: Rule, per-stratum counts and totals.
    """

    def __init__(self, index: IndexBook, settings: EncoderInputSettings) -> None:
        """Builds the split.

        Args:
            index: Labels lookup.
            settings: The notebook settings.
        """
        s = settings
        codes: dict[str, set[int]] = {}
        for scenario in s.scenarios:
            for vehicle, code in index.labels.get(scenario, {}).items():
                codes.setdefault(vehicle_key(scenario, vehicle), set()).add(code)
        self.class_of: dict[str, str] = {key: self.class_name(index, c) for key, c in codes.items()}
        strata: dict[tuple[str, str], list[str]] = {}
        for key, name in self.class_of.items():
            strata.setdefault((key.split(":")[0], name), []).append(key)
        rng = np.random.default_rng(s.split_seed)
        self.split_of: dict[str, str] = {}
        per_stratum = {}
        for (group, name), keys in sorted(strata.items()):
            ordered = sorted(keys)
            rng.shuffle(ordered)
            n_test = max(1, int(round(s.test_frac * len(ordered))))
            n_val = max(1, int(round(s.val_frac * len(ordered))))
            if n_test + n_val >= len(ordered):
                raise ValueError(f"stratum {group}/{name} has {len(ordered)} vehicles: no train vehicle left")
            for k, key in enumerate(ordered):
                if key in self.split_of:
                    raise ValueError(f"vehicle {key} is in two strata")  # RULE 3
                self.split_of[key] = "test" if k < n_test else ("val" if k < n_test + n_val else "train")
            per_stratum[f"{group}/{name}"] = {
                "train": len(ordered) - n_test - n_val,
                "val": n_val,
                "test": n_test,
                "total": len(ordered),
            }
        self.report = {
            "key": "<group>:<sender vehicle> (the T24 notebook's key): one split per key across all 4 families",
            "class_of_vehicle": "the set of known attack codes the key carries over the 4 families (-1 ignored)",
            "rule": (
                f"strata = (group, class of the vehicle), sorted; sorted keys shuffled by one numpy default_rng("
                f"{s.split_seed}) drawn stratum after stratum; test = first max(1, round({s.test_frac} n)), val = next "
                f"max(1, round({s.val_frac} n)), train = rest (the T48_tvt rule)"
            ),
            "per_stratum": per_stratum,
            "vehicles": {split: sum(v == split for v in self.split_of.values()) for split in s.vehicle_splits},
            "vehicle_ids": {
                split: sorted(k for k, v in self.split_of.items() if v == split) for split in s.vehicle_splits
            },
            "f1_note": (
                "the 4 families of a group replay the same benign traffic; one key = one split across them, so "
                "near-copies of a benign window in other families stay in its split (except F14 below)"
            ),
            "f14_limitation": (
                "GridSybil_0709 is a separate re-simulation of the 0709 traffic with different vehicle ids: a key "
                "shared with another 0709 family is not the same car, and GridSybil_0709 near-copies of cars of the "
                "other 0709 families can straddle splits (2.5-5.9% of test benign identities within 1 m in the T24 "
                "split). Not measured or fixed here; the 1416 group is not affected."
            ),
        }

    @staticmethod
    def class_name(index: IndexBook, codes: set[int]) -> str:
        """Class of a vehicle from its codes over the families: 'Benign', an attack name, or attack names joined."""
        known = sorted(c for c in codes if c != -1)
        if not known:
            return "Unknown"
        return "+".join(index.name(c) for c in known)

    def split(self, scenario: str, sender: int) -> str | None:
        """Returns the sender vehicle's split, or None if the vehicle has no split."""
        return self.split_of.get(vehicle_key(scenario, sender))


class AttackerRegistry:
    """Physical vehicles that are attackers anywhere in a scenario folder.

    Attributes:
        from_index: Scenario folder -> attacker vehicle ids from the `index.json` labels.
        from_files: Scenario folder -> attacker vehicle ids from the attack code in their own trace file name.
        file_code: (scenario folder, vehicle) -> the attack code in its own trace file name.
    """

    def __init__(self, index: IndexBook, settings: EncoderInputSettings) -> None:
        """Builds both attacker lists.

        Args:
            index: Labels lookup.
            settings: The notebook settings.

        Raises:
            ValueError: If a vehicle has two trace files in one scenario folder (the grouping would be ambiguous).
        """
        root = pathlib.Path(settings.prepared_dir)
        self.from_index: dict[str, set[int]] = {
            scenario: {v for v, c in index.labels.get(scenario, {}).items() if c not in (-1, settings.benign_code)}
            for scenario in settings.scenarios
        }
        self.from_files: dict[str, set[int]] = {scenario: set() for scenario in settings.scenarios}
        self.file_code: dict[tuple[str, int], int] = {}
        for scenario in settings.scenarios:
            for path in sorted((root / scenario).glob("*/traceJSON-*.json")):
                match = TRACE_NAME.match(path.name)
                if match is None:
                    raise ValueError(f"unexpected trace file name {path}")
                vehicle, code = int(match.group(1)), int(match.group(3))
                if (scenario, vehicle) in self.file_code:
                    raise ValueError(f"vehicle {vehicle} has two trace files in {scenario}")
                self.file_code[(scenario, vehicle)] = code
                if code != settings.benign_code:
                    self.from_files[scenario].add(vehicle)

    def is_attacker(self, scenario: str, vehicle: int) -> bool:
        """True if the vehicle is an attacker in the scenario folder by either source."""
        return vehicle in self.from_index.get(scenario, set()) or vehicle in self.from_files.get(scenario, set())

    def summary(self) -> dict[str, Any]:
        """Attacker counts per scenario folder and source, and where the sources disagree."""
        return {
            scenario: {
                "index_labels": len(self.from_index[scenario]),
                "trace_files": len(self.from_files[scenario]),
                "union": len(self.from_index[scenario] | self.from_files[scenario]),
                "only_in_index": len(self.from_index[scenario] - self.from_files[scenario]),
                "only_in_files": len(self.from_files[scenario] - self.from_index[scenario]),
            }
            for scenario in self.from_index
        }


INDEX = IndexBook(pathlib.Path(SETTINGS.prepared_dir) / "index.json")
SPLIT = VehicleSplit(INDEX, SETTINGS)
REGISTRY = AttackerRegistry(INDEX, SETTINGS)
_ids = SPLIT.report["vehicle_ids"]
assert sum(len(v) for v in _ids.values()) == len(SPLIT.split_of)  # RULE 3: every vehicle in exactly one split
assert not (set(_ids["train"]) & set(_ids["val"])) and not ((set(_ids["train"]) | set(_ids["val"])) & set(_ids["test"]))
_strata = pd.DataFrame(SPLIT.report["per_stratum"]).T
_strata["val_share"] = (_strata["val"] / _strata["total"]).round(4)
_strata["test_share"] = (_strata["test"] / _strata["total"]).round(4)
print(_strata.to_string())
print("attacker vehicles:", json.dumps(REGISTRY.summary()))
print(
    f"Takeaway: {len(SPLIT.split_of):,} vehicle keys split {SPLIT.report['vehicles']} over {len(_strata)} strata; "
    "⚠ RULE 3: this is a new 80 / 10 / 10 split over all 8 folders."
)

                                                                  train  val  test  total  val_share  test_share
0709/Benign                                                        3260  407   407   4074     0.0999      0.0999
0709/DataReplaySybil+DoSRandomSybil+DoSDisruptiveSybil              421   53    53    527     0.1006      0.1006
0709/GridSybil                                                      421   52    52    525     0.0990      0.0990
0709/GridSybil+DataReplaySybil+DoSRandomSybil+DoSDisruptiveSybil    556   69    69    694     0.0994      0.0994
1416/Benign                                                         948  118   118   1184     0.0997      0.0997
1416/GridSybil+DataReplaySybil+DoSRandomSybil+DoSDisruptiveSybil    405   51    51    507     0.1006      0.1006
attacker vehicles: {"DataReplaySybil_0709": {"index_labels": 1221, "trace_files": 1221, "union": 1221, "only_in_index": 0, "only_in_files": 0}, "DataReplaySybil_1416": {"index_labels": 507, "trace_files": 507,

## Reading, routing and cropping the prepared files

There is one prepared file per receiving car. `LinkCropper` reads one file and routes every link:

- empty link → dropped (`empty`); unknown sender label (code -1) → dropped (`unknown_label`);
- benign sender (code 0) that is an attacker in its folder by either source → dropped (`benign_attacker_sender`);
- benign sender otherwise → its vehicle's split (`train`, `val` or `test`);
- attack sender of a **test** vehicle → its evaluation split (`test_gridsybil`, …); of a train / val vehicle →
  dropped (`attack_train_val`).

A kept link is cut into non-overlapping crops of at most 32 messages. `count_file` counts (windows by length,
links, messages, vehicles, drops, the benign-train normalisation sums); `windows_of_file` normalises, pads to 32 and
serialises. `ParallelFiles` runs either over all files in `fork` worker processes, one batch of files at a time.

In [4]:
class TrainNormaliser:
    """Per-feature mean / std from the real rows of the benign train windows, built from streamed sums.

    The sums are taken around a fixed shift (the mean stored in `index.json`) only to keep the sum of squares
    numerically stable; the shift does not change the result.

    Attributes:
        features: Feature names, in column order.
        shift: Value subtracted before summing, shape (n_features,).
        mean: Per-feature train mean.
        std: Per-feature train standard deviation (1.0 for a constant feature).
        n_values: Per-feature number of train values used (missing log_dtau skipped).
    """

    def __init__(self, features: tuple[str, ...], shift: np.ndarray) -> None:
        self.features = list(features)
        self.shift = np.asarray(shift, dtype=np.float64)
        self.mean: np.ndarray | None = None
        self.std: np.ndarray | None = None
        self.n_values: np.ndarray | None = None

    def fit_from_sums(self, count: np.ndarray, total: np.ndarray, total_sq: np.ndarray) -> TrainNormaliser:
        """Sets mean / std from per-feature count, sum and sum of squares of (value - shift)."""
        if np.any(count == 0):
            raise ValueError("a feature has no train values; cannot fit the normaliser")
        centred_mean = total / count
        variance = np.maximum(total_sq / count - centred_mean**2, 0.0)
        self.n_values = count.astype(np.int64)
        self.mean = self.shift + centred_mean
        std = np.sqrt(variance)
        self.std = np.where(std > 0, std, 1.0)
        return self

    def fit_stand_in(self, mean: np.ndarray, std: np.ndarray) -> TrainNormaliser:
        """Uses given statistics (only for the size estimate, before the train statistics exist)."""
        self.mean, self.std, self.n_values = np.asarray(mean), np.asarray(std), np.zeros(len(self.features), int)
        return self

    def apply(self, x: np.ndarray) -> np.ndarray:
        """Normalises real rows; a missing value (the first log_dtau of a link) becomes 0, the train mean."""
        if self.mean is None:
            raise RuntimeError("fit the normaliser first")
        return np.nan_to_num((x - self.mean) / self.std, nan=0.0)

    def to_dict(self) -> dict[str, Any]:
        """The statistics as plain JSON-ready values."""
        return {
            "features": self.features,
            "mean": [float(v) for v in self.mean],
            "std": [float(v) for v in self.std],
            "n_values": [int(v) for v in self.n_values],
            "fitted_on": "benign train windows, real rows only (missing first log_dtau of a link skipped)",
            "applied_to": "every split, the attack evaluation splits included",
            "missing_log_dtau": "first message of each link has no previous message; stored as 0 (= train mean)",
            "padding": "padding rows are 0 in every feature; mask = 0 marks them",
        }


class LinkCropper:
    """Reads one prepared receiver file, routes its links to a split, and crops them into 32-row windows."""

    INFO_KEYS: tuple[str, ...] = (
        "family",
        "scenario",
        "run",
        "receiver_file",
        "receiver",
        "sender",
        "sender_pseudo",
        "link_window_index",
        "n_messages",
        "bucket",
        "label",
        "label_name",
        "is_attack",
    )

    def __init__(
        self,
        settings: EncoderInputSettings,
        index: IndexBook,
        split: VehicleSplit,
        registry: AttackerRegistry,
        normaliser: TrainNormaliser | None = None,
        route_all: bool = False,
    ) -> None:
        """Stores collaborators.

        Args:
            settings: The notebook settings.
            index: Labels lookup.
            split: Vehicle split lookup.
            registry: Attacker vehicles.
            normaliser: Fitted normaliser; needed only by `windows_of_file`.
            route_all: Keep the attack windows of every vehicle (size estimate only; never for the output).
        """
        self.settings = settings
        self.index = index
        self.split = split
        self.registry = registry
        self.normaliser = normaliser
        self.route_all = route_all
        self.attack_split = dict(settings.attack_splits)
        self.root = pathlib.Path(settings.prepared_dir)

    def route(self, scenario: str, sender: int, code: int, n_messages: int) -> tuple[str | None, str]:
        """Returns (output split, drop reason); exactly one of them is empty."""
        vehicle_split = self.split.split(scenario, sender)
        if n_messages == 0:
            return None, "empty"
        if code == -1:
            return None, "unknown_label"
        if vehicle_split is None:
            return None, "no_split"
        if code == self.settings.benign_code:
            if self.registry.is_attacker(scenario, sender):
                return None, "benign_attacker_sender"
            return vehicle_split, ""
        if code in self.attack_split:
            if vehicle_split == "test" or self.route_all:
                return self.attack_split[code], ""
            return None, "attack_train_val"
        return None, "other_code"

    def links(self, path_str: str) -> Iterator[tuple[dict, dict[str, Any]]]:
        """Yields (file facts, link facts) for every link of a file, dropped ones flagged with a reason."""
        path = pathlib.Path(path_str)
        record = json.loads(path.read_text())
        scenario = record["run"].split("/", 1)[0]
        facts = {
            "scenario": scenario,
            "family": family_of(scenario),
            "run": record["run"],
            "receiver_file": str(path.relative_to(self.root)),
            "receiver": int(record["receiver"]),
            "receiver_attack_code": int(record["receiver_attack_code"]),
            "file_name_code": self.registry.file_code.get((scenario, int(record["receiver"])), -2),
        }
        for link in record["links"]:
            sender = int(link["sender"])
            x = np.array(link["x"], dtype=np.float64).reshape(-1, N_FEATURES)  # JSON null -> NaN
            code = self.index.label(scenario, sender)
            out_split, reason = self.route(scenario, sender, code, len(x))
            yield facts, {
                "sender": sender,
                "sender_pseudo": int(link["sender_pseudo"]),
                "x": x,
                "code": code,
                "split": out_split,
                "drop": reason,
            }

    def count_file(self, path_str: str) -> dict[str, Any]:
        """Counts one file: windows by length, links, messages, vehicles, drops, train sums and quality tallies."""
        t = self.settings.seq_len
        shift = self.index.normalisation_mean
        out: dict[str, Any] = {
            "kept": {},
            "dropped": {},
            "seen": {},
            "train_count": np.zeros(N_FEATURES),
            "train_sum": np.zeros(N_FEATURES),
            "train_sum_sq": np.zeros(N_FEATURES),
            "nan_real_rows": np.zeros(N_FEATURES, dtype=np.int64),
            "nan_log_dtau_not_first": 0,
            "duplicate_links": 0,
            "receiver_code_mismatch": 0,
            "benign_with_attacker_receiver": 0,
        }
        seen_links: set[tuple[int, int]] = set()
        first = True
        for facts, link in self.links(path_str):
            if first:
                out["receiver_code_mismatch"] += facts["receiver_attack_code"] != facts["file_name_code"]
                first = False
            x, n_messages = link["x"], len(link["x"])
            name = self.index.name(link["code"]) if link["code"] != -1 else "Unknown"
            if n_messages:
                seen = out["seen"].setdefault((facts["scenario"], name), [0, 0])
                seen[0] += 1
                seen[1] += n_messages
            vehicle = vehicle_key(facts["scenario"], link["sender"])
            n_windows = -(-n_messages // t)
            if link["drop"]:
                tally = out["dropped"].setdefault(
                    (facts["scenario"], name, link["drop"]),
                    {"links": 0, "messages": 0, "windows": 0, "vehicles": set()},
                )
                tally["links"] += 1
                tally["messages"] += n_messages
                tally["windows"] += n_windows
                tally["vehicles"].add(vehicle)
                continue
            link_key = (link["sender_pseudo"], link["sender"])
            out["duplicate_links"] += link_key in seen_links
            seen_links.add(link_key)
            key = (link["split"], facts["scenario"], name)
            entry = out["kept"].setdefault(
                key, {"hist": np.zeros(t + 1, dtype=np.int64), "links": 0, "messages": 0, "vehicles": set()}
            )
            entry["hist"][t] += n_messages // t
            if n_messages % t:
                entry["hist"][n_messages % t] += 1
            entry["links"] += 1
            entry["messages"] += n_messages
            entry["vehicles"].add(vehicle)
            missing = np.isnan(x)
            out["nan_real_rows"] += missing.sum(axis=0)
            out["nan_log_dtau_not_first"] += int(missing[1:, LOG_DTAU_COL].sum())
            if link["code"] == self.settings.benign_code and facts["receiver_attack_code"] != self.settings.benign_code:
                out["benign_with_attacker_receiver"] += n_windows
            if link["split"] == "train":
                centred = x - shift
                present = ~missing
                out["train_count"] += present.sum(axis=0)
                out["train_sum"] += np.where(present, centred, 0.0).sum(axis=0)
                out["train_sum_sq"] += np.where(present, centred**2, 0.0).sum(axis=0)
        return out

    def windows_of_file(self, path_str: str) -> dict[str, dict[str, list]]:
        """Normalises, pads and serialises every window of one file, grouped by output split.

        Returns:
            split -> {"data": window JSON texts, "info": info JSON texts, "n": real rows per window}.
        """
        t, decimals = self.settings.seq_len, self.settings.decimals
        out: dict[str, dict[str, list]] = {}
        for facts, link in self.links(path_str):
            if link["drop"]:
                continue
            x = np.round(self.normaliser.apply(link["x"]), decimals) + 0.0  # -0.0 -> 0.0
            name = self.index.name(link["code"])
            bucket = out.setdefault(link["split"], {"data": [], "info": [], "n": []})
            for k, start in enumerate(range(0, len(x), self.settings.crop_stride)):
                rows = x[start : start + t]
                n = len(rows)
                if n < self.settings.min_messages:
                    continue
                padded = np.zeros((t, N_FEATURES))
                padded[:n] = rows
                mask = [1] * n + [0] * (t - n)
                window_id = f"{facts['receiver_file']}#{link['sender_pseudo']}-{link['sender']}#{k}"
                bucket["data"].append(
                    f'{{"id":{json.dumps(window_id)},"x":{json.dumps(padded.tolist(), separators=(",", ":"))},'
                    f'"mask":{json.dumps(mask, separators=(",", ":"))}}}'
                )
                info = {
                    "id": window_id,
                    "family": facts["family"],
                    "scenario": facts["scenario"],
                    "run": facts["run"],
                    "receiver_file": facts["receiver_file"],
                    "receiver": facts["receiver"],
                    "sender": link["sender"],
                    "sender_pseudo": link["sender_pseudo"],
                    "link_window_index": k,
                    "n_messages": n,
                    "bucket": t,
                    "label": link["code"],
                    "label_name": name,
                    "is_attack": int(link["code"] != self.settings.benign_code),
                }
                bucket["info"].append(json.dumps(info, separators=(",", ":")))
                bucket["n"].append(n)
        return out


_CROPPER: LinkCropper | None = None  # set by ParallelFiles before the workers fork


def count_worker(path_str: str) -> dict[str, Any]:
    """Worker entry point of the counting pass (must stay top level)."""
    return _CROPPER.count_file(path_str)


def windows_worker(path_str: str) -> dict[str, dict[str, list]]:
    """Worker entry point of the writing pass (must stay top level)."""
    return _CROPPER.windows_of_file(path_str)


class ParallelFiles:
    """Lists the prepared files and runs a worker over them in batches with a fork pool."""

    def __init__(self, settings: EncoderInputSettings) -> None:
        """Stores the settings.

        Args:
            settings: The notebook settings.
        """
        self.settings = settings
        self.root = pathlib.Path(settings.prepared_dir)

    def files(self, scenario: str | None = None) -> list[str]:
        """Prepared files of the chosen scenarios (or one scenario), sorted so the output order is reproducible."""
        found: list[str] = []
        for name in [scenario] if scenario else self.settings.scenarios:
            folder = self.root / name
            if not folder.is_dir():
                raise FileNotFoundError(f"missing scenario folder {folder}")
            found.extend(str(p) for p in sorted(folder.glob("*/traceJSON-*.json")))
        return found

    def run(self, cropper: LinkCropper, worker: Callable[[str], Any], paths: list[str]) -> Iterator[Any]:
        """Yields the worker's result for every path, in order, one batch of files at a time."""
        global _CROPPER
        _CROPPER = cropper
        context = multiprocessing.get_context("fork")
        with context.Pool(self.settings.workers) as pool:
            for start in range(0, len(paths), self.settings.files_per_batch):
                batch = paths[start : start + self.settings.files_per_batch]
                yield from pool.map(worker, batch, chunksize=4)


FILES = ParallelFiles(SETTINGS)
ALL_FILES = FILES.files()
assert len(ALL_FILES) == len(REGISTRY.file_code)
print(f"{len(ALL_FILES):,} prepared files in {len(SETTINGS.scenarios)} scenario folders")

23,032 prepared files in 8 scenario folders


## Counting pass: counts and the benign train normalisation

Every file is read once. Nothing is kept per window: only counts per split × scenario × class (windows by length,
links, messages, vehicles), the drops per reason, and the per-feature count, sum and sum of squares over the real
rows of the **benign train** windows. The per-scenario link and message totals (kept + dropped) are compared with
`index.json`.

In [5]:
class CountTotals:
    """Adds up the counting-pass results of all files."""

    def __init__(self, settings: EncoderInputSettings) -> None:
        """Starts empty totals.

        Args:
            settings: The notebook settings.
        """
        self.settings = settings
        self.kept: dict[tuple[str, str, str], dict[str, Any]] = {}
        self.dropped: dict[tuple[str, str, str], dict[str, Any]] = {}
        self.seen: dict[tuple[str, str], list[int]] = {}
        self.train_count = np.zeros(N_FEATURES)
        self.train_sum = np.zeros(N_FEATURES)
        self.train_sum_sq = np.zeros(N_FEATURES)
        self.nan_real_rows = np.zeros(N_FEATURES, dtype=np.int64)
        self.nan_log_dtau_not_first = 0
        self.duplicate_links = 0
        self.receiver_code_mismatch = 0
        self.benign_with_attacker_receiver = 0
        self.files = 0

    def add(self, result: dict[str, Any]) -> None:
        """Adds one file's counts."""
        self.files += 1
        t = self.settings.seq_len
        for key, part in result["kept"].items():
            entry = self.kept.setdefault(
                key, {"hist": np.zeros(t + 1, dtype=np.int64), "links": 0, "messages": 0, "vehicles": set()}
            )
            entry["hist"] += part["hist"]
            entry["links"] += part["links"]
            entry["messages"] += part["messages"]
            entry["vehicles"] |= part["vehicles"]
        for key, part in result["dropped"].items():
            entry = self.dropped.setdefault(key, {"links": 0, "messages": 0, "windows": 0, "vehicles": set()})
            for field in ("links", "messages", "windows"):
                entry[field] += part[field]
            entry["vehicles"] |= part["vehicles"]
        for key, (links, messages) in result["seen"].items():
            tally = self.seen.setdefault(key, [0, 0])
            tally[0] += links
            tally[1] += messages
        self.train_count += result["train_count"]
        self.train_sum += result["train_sum"]
        self.train_sum_sq += result["train_sum_sq"]
        self.nan_real_rows += result["nan_real_rows"]
        for field in (
            "nan_log_dtau_not_first",
            "duplicate_links",
            "receiver_code_mismatch",
            "benign_with_attacker_receiver",
        ):
            setattr(self, field, getattr(self, field) + result[field])

    def table(self) -> pd.DataFrame:
        """Windows, messages, links, vehicles, padding and very short windows per split × scenario × class."""
        t, short = self.settings.seq_len, self.settings.short_below
        lengths = np.arange(t + 1)
        rows = []
        for (split, scenario, name), entry in self.kept.items():
            hist = entry["hist"]
            windows = int(hist.sum())
            messages = int((hist * lengths).sum())
            if messages != entry["messages"]:
                raise AssertionError(f"window lengths do not add up to the messages of {split}/{scenario}/{name}")
            rows.append(
                {
                    "split": split,
                    "scenario": scenario,
                    "class": name,
                    "windows": windows,
                    "messages": messages,
                    "links": entry["links"],
                    "vehicles": len(entry["vehicles"]),
                    "windows_1_3": int(hist[1:short].sum()),
                }
            )
        table = pd.DataFrame(rows)
        table["split"] = pd.Categorical(table["split"], self.settings.all_splits, ordered=True)
        return table.sort_values(["split", "scenario", "class"]).set_index(["split", "scenario", "class"])

    def vehicles_by_split(self) -> dict[str, set[str]]:
        """Vehicle keys with at least one kept window, per output split."""
        result: dict[str, set[str]] = {}
        for (split, _, _), entry in self.kept.items():
            result.setdefault(split, set()).update(entry["vehicles"])
        return result

    def dropped_table(self) -> pd.DataFrame:
        """Links, messages, windows (as they would have been cut) and vehicles dropped, per scenario × class × reason."""
        rows = [
            {
                "scenario": scenario,
                "class": name,
                "reason": reason,
                **{f: e[f] for f in ("links", "messages", "windows")},
                "vehicles": len(e["vehicles"]),
            }
            for (scenario, name, reason), e in sorted(self.dropped.items())
        ]
        return pd.DataFrame(rows).set_index(["scenario", "class", "reason"])

    def compare_with_index(self, index: IndexBook) -> pd.DataFrame:
        """Links and messages per scenario × class seen here (kept + dropped) vs in `index.json`."""
        rows = []
        for scenario in self.settings.scenarios:
            for name, recorded in index.counts[scenario].items():
                here = self.seen.get((scenario, name), [0, 0])
                rows.append(
                    {
                        "scenario": scenario,
                        "class": name,
                        "links": here[0],
                        "links_index": recorded["links"],
                        "messages": here[1],
                        "messages_index": recorded["messages"],
                    }
                )
        table = pd.DataFrame(rows).set_index(["scenario", "class"])
        table["match"] = (table["links"] == table["links_index"]) & (table["messages"] == table["messages_index"])
        return table


_start = time.perf_counter()
COUNTS = CountTotals(SETTINGS)
for _result in FILES.run(LinkCropper(SETTINGS, INDEX, SPLIT, REGISTRY), count_worker, ALL_FILES):
    COUNTS.add(_result)
COUNT_SECONDS = time.perf_counter() - _start
NORMALISER = TrainNormaliser(FEATURE_NAMES, INDEX.normalisation_mean).fit_from_sums(
    COUNTS.train_count, COUNTS.train_sum, COUNTS.train_sum_sq
)
INDEX_CHECK = COUNTS.compare_with_index(INDEX)
TABLE = COUNTS.table()
DROPPED = COUNTS.dropped_table()
_vehicles = COUNTS.vehicles_by_split()
_other_nan = {f: int(n) for f, n in zip(FEATURE_NAMES, COUNTS.nan_real_rows) if n and f != "log_dtau"}
assert not _other_nan, f"missing values outside log_dtau: {_other_nan}"
assert COUNTS.nan_log_dtau_not_first == 0, "a missing log_dtau that is not the first message of its link"
assert COUNTS.duplicate_links == 0, "two links with the same (pseudonym, sender) in one file: window ids would clash"
assert COUNTS.receiver_code_mismatch == 0, "receiver_attack_code differs from the code in the trace file name"
assert not any(reason in ("no_split", "other_code") for _, _, reason in COUNTS.dropped), DROPPED
assert INDEX_CHECK["match"].all(), INDEX_CHECK[~INDEX_CHECK["match"]]
for _a, _b in (("train", "val"), ("train", "test"), ("val", "test")):
    assert not (_vehicles.get(_a, set()) & _vehicles.get(_b, set())), f"a vehicle in {_a} and {_b}"  # RULE 3
for _split in dict(SETTINGS.attack_splits).values():
    assert all(SPLIT.split_of[v] == "test" for v in _vehicles.get(_split, set())), _split
_benign_splits = set(SETTINGS.vehicle_splits)
assert set(TABLE.reset_index().query("split in @_benign_splits")["class"]) == {"Benign"}
print(f"counted {COUNTS.files:,} files in {COUNT_SECONDS:.1f} s")
print(INDEX_CHECK.to_string())
print("\ndropped:\n" + DROPPED.to_string())
print(
    pd.DataFrame(
        {"mean": NORMALISER.mean, "std": NORMALISER.std, "n_values": NORMALISER.n_values}, index=FEATURE_NAMES
    ).round(4)
)
print(
    f"Takeaway: {int(TABLE['windows'].sum()):,} windows from {int(TABLE['links'].sum()):,} kept links; "
    f"totals match index.json; normalisation from {int(NORMALISER.n_values[0]):,} benign train rows."
)

counted 23,032 files in 6.4 s
                                              links  links_index  messages  messages_index  match
scenario                class                                                                    
DataReplaySybil_0709    Benign               103860       103860   2081685         2081685   True
                        DataReplaySybil      588828       588828    873068          873068   True
DataReplaySybil_1416    Benign                21658        21658    528518          528518   True
                        DataReplaySybil      143829       143829    239440          239440   True
DoSDisruptiveSybil_0709 Benign               103859       103859   2081446         2081446   True
                        DoSDisruptiveSybil  1547646      1547646   1741441         1741441   True
DoSDisruptiveSybil_1416 Benign                21688        21688    528397          528397   True
                        DoSDisruptiveSybil   381293       381293    477450          4774

## Windows per split, padding and very short windows

Padding share = padding rows / stored rows (every window stores 32 rows). The 1–3-row share is the F11 context:
DataReplay / DoS attackers change pseudonym every 1–2 messages, so their windows are mostly 1–3 rows long, and
window length alone would separate them from benign windows.

In [6]:
class SplitReport:
    """Per-split summaries of the counting-pass table."""

    def __init__(self, table: pd.DataFrame, settings: EncoderInputSettings) -> None:
        """Stores the table.

        Args:
            table: `CountTotals.table()`.
            settings: The notebook settings.
        """
        self.table = table
        self.settings = settings

    def summarise(self, part: pd.DataFrame) -> dict[str, float | int]:
        """Windows, messages, links, padding share and 1–3-row share of a slice of the table."""
        windows = int(part["windows"].sum())
        messages = int(part["messages"].sum())
        return {
            "windows": windows,
            "messages": messages,
            "links": int(part["links"].sum()),
            "padding_share": round(1.0 - messages / (windows * self.settings.seq_len), 4) if windows else 0.0,
            "windows_1_3": int(part["windows_1_3"].sum()),
            "share_1_3": round(float(part["windows_1_3"].sum()) / windows, 4) if windows else 0.0,
        }

    def per_split(self, vehicles: dict[str, set[str]]) -> pd.DataFrame:
        """One row per output split, with its vehicle count."""
        rows = {}
        for split in self.settings.all_splits:
            part = self.table.xs(split, level="split") if split in self.table.index.get_level_values(0) else None
            rows[split] = (self.summarise(part) if part is not None else {}) | {
                "vehicles": len(vehicles.get(split, set()))
            }
        return pd.DataFrame.from_dict(rows, orient="index")

    def per_split_scenario_class(self) -> dict[str, dict[str, dict[str, dict]]]:
        """split -> scenario -> class -> counts, padding share and 1–3-row share."""
        result: dict[str, dict[str, dict[str, dict]]] = {}
        for (split, scenario, name), row in self.table.iterrows():
            frame = row.to_frame().T
            result.setdefault(str(split), {}).setdefault(scenario, {})[name] = self.summarise(frame) | {
                "vehicles": int(row["vehicles"])
            }
        return result


REPORT = SplitReport(TABLE, SETTINGS)
PER_SPLIT = REPORT.per_split(_vehicles)
print(TABLE.to_string())
print()
print(PER_SPLIT.to_string())
_overall = REPORT.summarise(TABLE)
print(
    f"Takeaway: {_overall['windows']:,} windows, padding {_overall['padding_share']:.1%}; benign train / val / test = "
    f"{int(PER_SPLIT.loc['train', 'windows']):,} / {int(PER_SPLIT.loc['val', 'windows']):,} / "
    f"{int(PER_SPLIT.loc['test', 'windows']):,} windows."
)

                                                               windows  messages   links  vehicles  windows_1_3
split              scenario                class                                                               
train              DataReplaySybil_0709    Benign               102141   1671696   83127      2280        14120
                   DataReplaySybil_1416    Benign                23513    418994   17380       946         3110
                   DoSDisruptiveSybil_0709 Benign               102127   1671339   83100      2280        14110
                   DoSDisruptiveSybil_1416 Benign                23558    419051   17408       946         3140
                   DoSRandomSybil_0709     Benign               102127   1671339   83100      2280        14110
                   DoSRandomSybil_1416     Benign                23558    419051   17408       946         3140
                   GridSybil_0709          Benign               102225   1676418   83044      2276      

## Size estimate before anything is written

A sample of files per scenario is serialised exactly as the writer will do it (attack links of every vehicle
included, normalised with the train statistics) and the plain-JSON bytes per window are measured per class. They
are scaled to the counted windows per class. If the projection exceeds 4 GB, the shards are gzipped
(`.json.gz`); the guard stops here if the projection does not fit the free space minus 5 GB.

In [7]:
class SizeEstimator:
    """Projects the output size from a sample of serialised files."""

    def __init__(self, settings: EncoderInputSettings, files: ParallelFiles) -> None:
        """Stores collaborators.

        Args:
            settings: The notebook settings.
            files: File lister and parallel runner.
        """
        self.settings = settings
        self.files = files

    def sample(self) -> list[str]:
        """Evenly spaced files of every scenario."""
        picked = []
        for scenario in self.settings.scenarios:
            paths = self.files.files(scenario)
            step = max(1, len(paths) // self.settings.size_sample_files)
            picked += paths[::step][: self.settings.size_sample_files]
        return picked

    def bytes_per_window(self, cropper: LinkCropper) -> pd.DataFrame:
        """Mean plain and gzipped bytes per window (data + info) for each output split's class."""
        data: dict[str, list[str]] = {}
        info: dict[str, list[str]] = {}
        for result in self.files.run(cropper, windows_worker, self.sample()):
            for split, part in result.items():
                name = "benign" if split in self.settings.vehicle_splits else split
                data.setdefault(name, []).extend(part["data"])
                info.setdefault(name, []).extend(part["info"])
        rows = {}
        for name in data:
            plain = len(",".join(data[name]).encode()) + len(",".join(info[name]).encode())
            packed = len(gzip.compress(",".join(data[name]).encode(), 6)) + len(
                gzip.compress(",".join(info[name]).encode(), 6)
            )
            rows[name] = {
                "sample_windows": len(data[name]),
                "plain": plain / len(data[name]),
                "gzip": packed / len(data[name]),
            }
        return pd.DataFrame(rows).T

    def project(self, per_window: pd.DataFrame, per_split: pd.DataFrame) -> pd.DataFrame:
        """Projected plain and gzipped bytes per output split."""
        rows = {}
        for split, row in per_split.iterrows():
            name = "benign" if split in self.settings.vehicle_splits else split
            rows[split] = {
                "windows": int(row["windows"]),
                "plain_gb": row["windows"] * per_window.loc[name, "plain"] / 1e9,
                "gzip_gb": row["windows"] * per_window.loc[name, "gzip"] / 1e9,
            }
        return pd.DataFrame(rows).T


_estimator = SizeEstimator(SETTINGS, FILES)
_per_window = _estimator.bytes_per_window(LinkCropper(SETTINGS, INDEX, SPLIT, REGISTRY, NORMALISER, route_all=True))
SIZE = _estimator.project(_per_window, PER_SPLIT)
PLAIN_BYTES = float(SIZE["plain_gb"].sum() * 1e9 * 1.10)  # 10% safety on top of the sample projection
USE_GZIP = PLAIN_BYTES > SETTINGS.gzip_above_gb * 1e9
PROJECTED_BYTES = float(SIZE["gzip_gb"].sum() * 1e9 * 1.10) if USE_GZIP else PLAIN_BYTES
EXISTING_BYTES = folder_bytes(pathlib.Path(SETTINGS.out_dir)) if SETTINGS.overwrite else 0
print(_per_window.round(1).to_string())
print(SIZE.round(3).to_string())
GUARD.check(PROJECTED_BYTES, already_there=EXISTING_BYTES)
print(
    f"Takeaway: plain JSON projected at {PLAIN_BYTES / 1e9:.2f} GB (incl. 10% safety) -> "
    f"{'gzipped shards (.json.gz)' if USE_GZIP else 'plain JSON shards (the T48_benign layout)'}, "
    f"{PROJECTED_BYTES / 1e9:.2f} GB; {(GUARD.allowance() + EXISTING_BYTES) / 1e9:.1f} GB may be written."
)

                    sample_windows   plain   gzip
benign                      4409.0  3159.5  516.4
test_datareplay             4693.0  2515.4   76.6
test_dosdisruptive         12154.0  2520.7   61.8
test_dosrandom             12155.0  2489.5   69.5
test_gridsybil              1901.0  3061.6  405.8
                     windows  plain_gb  gzip_gb
train               502773.0     1.589    0.260
val                  65313.0     0.206    0.034
test                 60142.0     0.190    0.031
test_gridsybil       28087.0     0.086    0.011
test_datareplay      76283.0     0.192    0.006
test_dosrandom      199834.0     0.497    0.014
test_dosdisruptive  199826.0     0.504    0.012
Takeaway: plain JSON projected at 3.59 GB (incl. 10% safety) -> plain JSON shards (the T48_benign layout), 3.59 GB; 117.6 GB may be written.


## Writing pass: normalise, pad and stream the shards

Every file is read a second time. Each window is normalised with the **benign train** statistics, rounded to 4
decimals, padded to 32 rows and appended to its split's buffer; a buffer is written as soon as it holds 10,000
windows. Layout (the `T48_benign` layout, so existing loaders read it):

- `<split>/b32/part-XXXXX.json` = `{"seq_len": 32, "max_seq_len": 32, "features": [...], "windows": [{"id", "x",
  "mask"}]}`, x 32 × 13. **No labels and no vehicle ids.**
- `<split>/b32/part-XXXXX_info.json` = the bookkeeping for the same windows, in the same order (family, scenario,
  run, receiver file, receiver, sender, sender pseudonym, window index in the link, n_messages, bucket, label,
  label name, is_attack).

The writer only writes under `src/data/encoder_input/all/` and stops if the free space drops below the margin.

In [8]:
class ShardedSplitWriter:
    """Streams serialised windows into per-split shards of a fixed size."""

    def __init__(self, settings: EncoderInputSettings, guard: DiskGuard, use_gzip: bool) -> None:
        """Prepares the output folder.

        Args:
            settings: The notebook settings.
            guard: Disk guard, checked before every shard.
            use_gzip: Write `.json.gz` shards instead of plain JSON.
        """
        self.settings = settings
        self.guard = guard
        self.use_gzip = use_gzip
        self.out_dir = pathlib.Path(settings.out_dir).resolve()
        root = pathlib.Path(settings.allowed_out_root).resolve()
        if not self.out_dir.is_relative_to(root):
            raise PermissionError(f"refusing to write to {self.out_dir}: outside {root}")
        self.buffers: dict[str, dict[str, list]] = {}
        self.numbers: dict[str, int] = {}
        self.shards: list[dict[str, Any]] = []
        self.header = (
            f'{{"seq_len":{settings.seq_len},"max_seq_len":{settings.seq_len},'
            f'"features":{json.dumps(list(FEATURE_NAMES), separators=(",", ":"))},"windows":['
        )

    def prepare_folder(self) -> None:
        """Creates the output folder; clears a previous output only when overwrite is set."""
        if self.out_dir.exists() and any(self.out_dir.iterdir()):
            if not self.settings.overwrite:
                raise FileExistsError(f"{self.out_dir} is not empty; set overwrite=True to replace it")
            shutil.rmtree(self.out_dir)
        self.out_dir.mkdir(parents=True, exist_ok=True)

    def add(self, split: str, part: dict[str, list]) -> None:
        """Appends one file's windows of a split; writes full shards."""
        buffer = self.buffers.setdefault(split, {"data": [], "info": [], "n": []})
        for field in ("data", "info", "n"):
            buffer[field].extend(part[field])
        while len(buffer["data"]) >= self.settings.shard_size:
            self.flush(split, self.settings.shard_size)

    def flush(self, split: str, count: int | None = None) -> None:
        """Writes the first `count` buffered windows of a split (all when None) as one shard."""
        buffer = self.buffers[split]
        count = len(buffer["data"]) if count is None else count
        if count == 0:
            return
        if self.guard.allowance() < 0:
            raise RuntimeError("free space dropped below the margin while writing; stopped")
        number = self.numbers.get(split, 0)
        self.numbers[split] = number + 1
        relative = f"{split}/b{self.settings.seq_len:02d}"
        (self.out_dir / relative).mkdir(parents=True, exist_ok=True)
        suffix = ".json.gz" if self.use_gzip else ".json"
        name = f"part-{number:05d}"
        data_text = self.header + ",".join(buffer["data"][:count]) + "]}"
        info_text = (
            f'{{"split":{json.dumps(split)},"bucket":{self.settings.seq_len},"shard":"{name}","windows":['
            + ",".join(buffer["info"][:count])
            + "]}"
        )
        self.write_text(self.out_dir / relative / f"{name}{suffix}", data_text)
        self.write_text(self.out_dir / relative / f"{name}_info{suffix}", info_text)
        self.shards.append(
            {
                "split": split,
                "bucket": self.settings.seq_len,
                "file": f"{relative}/{name}{suffix}",
                "info": f"{relative}/{name}_info{suffix}",
                "windows": count,
                "messages": int(sum(buffer["n"][:count])),
            }
        )
        for field in ("data", "info", "n"):
            del buffer[field][:count]

    def write_text(self, path: pathlib.Path, text: str) -> None:
        """Writes one shard file, gzipped or plain."""
        if self.use_gzip:
            with gzip.open(path, "wt", compresslevel=6) as handle:
                handle.write(text)
        else:
            path.write_text(text)

    def close(self) -> None:
        """Writes what is left in every buffer."""
        for split in self.settings.all_splits:
            if split in self.buffers:
                self.flush(split)


_start = time.perf_counter()
WRITER = ShardedSplitWriter(SETTINGS, GUARD, USE_GZIP)
WRITER.prepare_folder()
for _result in FILES.run(LinkCropper(SETTINGS, INDEX, SPLIT, REGISTRY, NORMALISER), windows_worker, ALL_FILES):
    for _split, _part in _result.items():
        WRITER.add(_split, _part)
WRITER.close()
WRITE_SECONDS = time.perf_counter() - _start
SHARDS = pd.DataFrame(WRITER.shards)
_written = SHARDS.groupby("split").agg(
    shards=("file", "size"), windows=("windows", "sum"), messages=("messages", "sum")
)
print(_written.to_string())
for _split, _row in _written.iterrows():
    assert _row["windows"] == PER_SPLIT.loc[_split, "windows"] and _row["messages"] == PER_SPLIT.loc[_split, "messages"]
OUTPUT_BYTES = folder_bytes(WRITER.out_dir)
print(
    f"Takeaway: wrote {len(SHARDS)} shards, {OUTPUT_BYTES / 1e9:.2f} GB, in {WRITE_SECONDS:.0f} s; counts match the counting pass."
)

                    shards  windows  messages
split                                        
test                     7    60142    996650
test_datareplay          8    76283    114133
test_dosdisruptive      20   199826    227491
test_dosrandom          20   199834    227491
test_gridsybil           3    28087    405921
train                   51   502773   8365846
val                      7    65313   1083336
Takeaway: wrote 116 shards, 3.28 GB, in 13 s; counts match the counting pass.


## Write `metadata.json`

Rules (split, benign-only, evaluation splits, normalisation), settings, the vehicle split (per stratum, vehicle ids
per split), counts per split × scenario × class (windows, messages, vehicles, links, padding share, 1–3-row share),
per split totals, the excluded counts, the normalisation statistics, F1 / F14 notes, the shard list, the source and
the provenance.

In [9]:
class MetadataWriter:
    """Builds and writes metadata.json for the encoder input folder."""

    def __init__(self, settings: EncoderInputSettings) -> None:
        self.settings = settings

    @staticmethod
    def provenance() -> dict[str, Any]:
        """Git commit / dirty flag, library versions and a UTC timestamp."""

        def git(*args: str) -> str:
            try:
                return subprocess.run(["git", *args], capture_output=True, text=True, check=True).stdout.strip()
            except (OSError, subprocess.CalledProcessError):
                return ""

        return {
            "notebook": "src/pipeline/all/encoder_input_T32_benign.ipynb",
            "git_commit": git("rev-parse", "HEAD") or None,
            "git_dirty": bool(git("status", "--porcelain")),
            "python": platform.python_version(),
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "platform": platform.platform(),
            "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        }

    @staticmethod
    def plain(value: Any) -> Any:
        """Turns numpy scalars into plain Python values for JSON."""
        return value.item() if isinstance(value, np.generic) else value

    def write(self, out_dir: pathlib.Path, content: dict[str, Any]) -> dict[str, Any]:
        """Writes metadata.json and returns its content."""
        with open(out_dir / "metadata.json", "w") as handle:
            json.dump(content, handle, indent=1, default=self.plain)
        return content


_excluded = {
    reason: {
        "links": int(part["links"].sum()),
        "messages": int(part["messages"].sum()),
        "windows": int(part["windows"].sum()),
        "vehicles": int(part["vehicles"].sum()),
        "by_scenario_class": {
            f"{scenario}/{name}": {k: int(v) for k, v in row.items()}
            for (scenario, name), row in part.droplevel("reason").iterrows()
        },
    }
    for reason, part in DROPPED.groupby(level="reason")
}
for _reason in ("benign_attacker_sender", "attack_train_val", "unknown_label", "empty"):
    _excluded.setdefault(_reason, {"links": 0, "messages": 0, "windows": 0, "vehicles": 0, "by_scenario_class": {}})
_excluded["_note"] = (
    "vehicles = sum over scenario folders of distinct vehicle keys; windows = how many windows the dropped links "
    "would have given"
)
_s = SETTINGS
METADATA = MetadataWriter(SETTINGS).write(
    WRITER.out_dir,
    {
        "dataset": "all 8 scenario folders, T = 32, benign only (train / val / test) + attack windows of the test vehicles",
        "features": list(FEATURE_NAMES),
        "max_seq_len": _s.seq_len,
        "buckets": [_s.seq_len],
        "format": "gzipped JSON (.json.gz)" if USE_GZIP else "plain JSON (the T48_benign layout)",
        "layout": (
            f"<split>/b{_s.seq_len}/part-XXXXX{'.json.gz' if USE_GZIP else '.json'} (id, x {_s.seq_len} x 13 "
            "normalised, 4 decimals, mask; no labels) + part-XXXXX_info (same order: family, scenario, run, "
            "receiver_file, receiver, sender, sender_pseudo, link_window_index, n_messages, bucket, label, label_name, "
            "is_attack)"
        ),
        "rules": {
            "crop": (
                f"each link cut into consecutive non-overlapping crops of at most {_s.seq_len} messages (stride "
                f"{_s.crop_stride}); the last one may be short; every crop padded to {_s.seq_len} rows with a prefix "
                f"mask; min_messages {_s.min_messages} (every message of a kept link in exactly one window)"
            ),
            "split": "⚠ RULE 3: new 80 / 10 / 10 split by physical vehicle over all 8 folders; " + SPLIT.report["rule"],
            "benign_only": (
                f"train / val / test keep links whose sender has code {_s.benign_code} in its scenario folder and is not "
                "an attacker there by the index.json labels or the attack code of its own trace file"
            ),
            "evaluation_splits": (
                "attack windows of the test vehicles: "
                + ", ".join(f"{name} = code {code}" for code, name in _s.attack_splits)
                + "; attack windows of train / val vehicles are dropped"
            ),
            "normalisation": "mean / std of the real rows of the benign train windows, applied to every split",
            "window_id": "<receiver_file>#<sender_pseudo>-<sender>#<window index in the link>",
        },
        "settings": {
            k: ([list(p) for p in v] if k == "attack_splits" else list(v) if isinstance(v, tuple) else v)
            for k, v in dataclasses.asdict(SETTINGS).items()
        },
        "normalisation": NORMALISER.to_dict(),
        "split": {k: v for k, v in SPLIT.report.items()},
        "attackers": REGISTRY.summary(),
        "counts_by_split": {
            split: {k: (float(v) if k in ("padding_share", "share_1_3") else int(v)) for k, v in row.items()}
            for split, row in PER_SPLIT.iterrows()
        },
        "counts_by_split_scenario_class": REPORT.per_split_scenario_class(),
        "excluded": _excluded,
        "benign_windows_with_attacker_receiver": int(COUNTS.benign_with_attacker_receiver),
        "padding": {"overall": _overall["padding_share"], "share_1_3_overall": _overall["share_1_3"]},
        "f11_note": (
            "DataReplay / DoS attackers rotate pseudonyms every 1-2 messages, so most of their windows hold 1-3 real "
            "rows; window length alone separates them from benign windows (see share_1_3 per split)"
        ),
        "f1_note": SPLIT.report["f1_note"] + "; all family copies are kept and each window records its family",
        "f14_limitation": SPLIT.report["f14_limitation"],
        "index_check": "links and messages per scenario x class (kept + dropped) equal index.json counts",
        "size_bytes": OUTPUT_BYTES,
        "runtime_seconds": {"counting": round(COUNT_SECONDS, 1), "writing": round(WRITE_SECONDS, 1)},
        "shards": WRITER.shards,
        "source": {"prepared_dir": _s.prepared_dir, "scenarios": list(_s.scenarios), "index": INDEX.source},
        "provenance": MetadataWriter.provenance(),
    },
)
print(json.dumps(METADATA["counts_by_split"], indent=1))
print(f"Takeaway: metadata.json written ({len(METADATA['shards'])} shards listed).")

{
 "train": {
  "windows": 502773,
  "messages": 8365846,
  "links": 401986,
  "padding_share": 0.48,
  "windows_1_3": 68836,
  "share_1_3": 0.1369,
  "vehicles": 4206
 },
 "val": {
  "windows": 65313,
  "messages": 1083336,
  "links": 52108,
  "padding_share": 0.4817,
  "windows_1_3": 9102,
  "share_1_3": 0.1394,
  "vehicles": 524
 },
 "test": {
  "windows": 60142,
  "messages": 996650,
  "links": 48188,
  "padding_share": 0.4821,
  "windows_1_3": 8369,
  "share_1_3": 0.1392,
  "vehicles": 525
 },
 "test_gridsybil": {
  "windows": 28087,
  "messages": 405921,
  "links": 23166,
  "padding_share": 0.5484,
  "windows_1_3": 5722,
  "share_1_3": 0.2037,
  "vehicles": 172
 },
 "test_datareplay": {
  "windows": 76283,
  "messages": 114133,
  "links": 76283,
  "padding_share": 0.9532,
  "windows_1_3": 72400,
  "share_1_3": 0.9491,
  "vehicles": 173
 },
 "test_dosrandom": {
  "windows": 199834,
  "messages": 227491,
  "links": 199834,
  "padding_share": 0.9644,
  "windows_1_3": 199311,
  "shar

## Read back and check the written files

Every shard is re-opened (in worker processes) and checked: `x` is 32 × 13 with no missing value, `mask` is a
prefix of ones whose sum equals `n_messages`, padding rows are zero, ids match the info file, feature shards hold
only `id`, `x`, `mask`. Across shards: window ids are unique, no vehicle key is in two of train / val / test, the
vehicles of every `test_*` split are test vehicles, benign splits hold only label 0 and each `test_*` split only
its attack code, and the totals equal the counting pass.

In [10]:
def read_text(path: pathlib.Path) -> str:
    """Reads a shard file, gzipped or plain."""
    if path.suffix == ".gz":
        with gzip.open(path, "rt") as handle:
            return handle.read()
    return path.read_text()


def check_shard_worker(entry: dict[str, Any]) -> dict[str, Any]:
    """Checks one shard against its info file (worker function, must stay top level)."""
    out_dir = pathlib.Path(SETTINGS.out_dir)
    t = SETTINGS.seq_len
    data = json.loads(read_text(out_dir / entry["file"]))
    info = json.loads(read_text(out_dir / entry["info"]))["windows"]
    assert data["seq_len"] == t and data["features"] == list(FEATURE_NAMES), entry["file"]
    assert len(data["windows"]) == len(info) == entry["windows"], entry
    keys, ids, vehicles, labels, messages, short = set(), [], set(), set(), 0, 0
    for window, row in zip(data["windows"], info):
        keys |= set(window)
        x = np.asarray(window["x"], dtype=np.float64)
        mask = np.asarray(window["mask"], dtype=np.int64)
        n = row["n_messages"]
        assert window["id"] == row["id"], (window["id"], row["id"])
        assert x.shape == (t, N_FEATURES) and mask.shape == (t,), window["id"]
        assert not np.isnan(x).any(), f"missing value in {window['id']}"
        assert 1 <= n <= t and int(mask.sum()) == n and (mask[:n] == 1).all(), f"bad mask in {window['id']}"
        assert np.all(x[n:] == 0.0), f"non-zero padding in {window['id']}"
        ids.append(row["id"])
        vehicles.add(vehicle_key(row["scenario"], row["sender"]))
        labels.add(row["label"])
        messages += n
        short += n < SETTINGS.short_below
    return {
        "split": entry["split"],
        "keys": keys,
        "ids": ids,
        "vehicles": vehicles,
        "labels": labels,
        "messages": messages,
        "short": short,
    }


_start = time.perf_counter()
with multiprocessing.get_context("fork").Pool(SETTINGS.workers) as _pool:
    _checked = _pool.map(check_shard_worker, WRITER.shards, chunksize=2)
_ids_seen: set[str] = set()
_n_ids = 0
_by_split: dict[str, dict[str, Any]] = {}
for _c in _checked:
    assert _c["keys"] == {"id", "x", "mask"}, _c["keys"]  # no labels in feature shards
    _n_ids += len(_c["ids"])
    _ids_seen.update(_c["ids"])
    _agg = _by_split.setdefault(
        _c["split"], {"windows": 0, "messages": 0, "short": 0, "vehicles": set(), "labels": set()}
    )
    _agg["windows"] += len(_c["ids"])
    _agg["messages"] += _c["messages"]
    _agg["short"] += _c["short"]
    _agg["vehicles"] |= _c["vehicles"]
    _agg["labels"] |= _c["labels"]
assert _n_ids == len(_ids_seen), f"{_n_ids - len(_ids_seen)} duplicate window ids"
for _a, _b in (("train", "val"), ("train", "test"), ("val", "test")):
    assert not (_by_split[_a]["vehicles"] & _by_split[_b]["vehicles"]), f"vehicle in {_a} and {_b}"  # RULE 3
for _split in SETTINGS.vehicle_splits:
    assert _by_split[_split]["labels"] == {SETTINGS.benign_code}, (_split, _by_split[_split]["labels"])
    assert all(SPLIT.split_of[v] == _split for v in _by_split[_split]["vehicles"]), _split
for _code, _split in SETTINGS.attack_splits:
    assert _by_split[_split]["labels"] == {_code}, (_split, _by_split[_split]["labels"])
    assert _by_split[_split]["vehicles"] <= set(SPLIT.report["vehicle_ids"]["test"]), _split
for _split, _agg in _by_split.items():
    assert _agg["windows"] == PER_SPLIT.loc[_split, "windows"] and _agg["messages"] == PER_SPLIT.loc[_split, "messages"]
    assert _agg["short"] == PER_SPLIT.loc[_split, "windows_1_3"], _split
print(f"checked {len(WRITER.shards)} shards in {time.perf_counter() - _start:.0f} s")
print(
    pd.DataFrame(
        {
            s: {
                "windows": a["windows"],
                "messages": a["messages"],
                "vehicles": len(a["vehicles"]),
                "labels": sorted(a["labels"]),
            }
            for s, a in _by_split.items()
        }
    ).T.to_string()
)
print(
    f"Takeaway: all checks passed: {_n_ids:,} unique windows, no vehicle in two of train / val / test, labels as "
    f"intended; total notebook time {time.perf_counter() - _NOTEBOOK_START:.0f} s."
)

checked 116 shards in 7 s
                   windows messages vehicles labels
train               502773  8365846     4206    [0]
test_datareplay      76283   114133      173   [17]
val                  65313  1083336      524    [0]
test                 60142   996650      525    [0]
test_dosdisruptive  199826   227491      173   [19]
test_dosrandom      199834   227491      173   [18]
test_gridsybil       28087   405921      172   [16]
Takeaway: all checks passed: 1,132,258 unique windows, no vehicle in two of train / val / test, labels as intended; total notebook time 28 s.
